# 第102章 梯度提升与加法模型

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 17 / 34 步：从模型分数走向业务评价与阈值**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 随机森林回归  →  **本章任务：** 梯度提升与加法模型  →  **下一步：** 多分类与Softmax
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：当我们要预测销量、房价这类连续数值时，单个模型常常"偏科"——要么欠拟合、要么过拟合。梯度提升的思路是把很多个弱小的决策树连起来，每一棵都去修补前面模型的漏洞，最终合出一棵更稳、更准的整体模型。本章用糖尿病数据集带你把"加法更新、负梯度、学习率、早停"这些概念，落到一段能跑的代码上。


## 本章目标

学完本章，你将能够：

- **理解**：理解「梯度提升与加法模型」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「梯度提升与加法模型」的关键输出指标。
- **迁移**：能把「梯度提升与加法模型」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：加法模型是另一条与“多树投票”不同的路：我这一棵树学完，把没猜对的部分交给下一棵树去“接力补救”，一轮轮累加（加上一个缩小的更新），慢慢逼近更强的预测。学习率和迭代轮数就是这场“接力”的步幅与棒数，决定了走得稳还是走得快。


- 加法更新：\(F_m(x)=F_{m-1}(x)+\eta h_m(x)\)
- 负梯度给出下一轮拟合方向
- 较小学习率通常需要更多迭代（打个比方：走得慢（学习率小）就要多走几步，但每步都稳；想一步迈到位的学得太快，反而容易跨过头、越学越偏。）
- 验证集早停属于模型选择的一部分


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | 参见本节示例 | 先明确样本、特征、目标和验证方式，再训练模型。 | 同时增大学习率和模型复杂度 |
| 模型、公式与诊断 | `rows.append()`、`m.predict()`、`pd.DataFrame()`、`result.round()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 在测试集上进行大量参数尝试 |


## 例 1｜数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-102 -->
### 数学推导｜提升模型逐步拟合残差

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜从简单初始模型开始。** 例如平方损失下 $F_0(x)$ 可取目标均值。

**第 2 步｜计算当前模型还没解释的方向。** 一般损失下使用负梯度

$$
r_{im}=-\left.\frac{\partial L(y_i,F(x_i))}{\partial F(x_i)}\right|_{F=F_{m-1}}
$$

平方损失时它正比于普通残差 $y_i-F_{m-1}(x_i)$。

**第 3 步｜让新弱学习器拟合该方向并更新。** $F_m=F_{m-1}+\eta h_m$；递推展开后就是多个弱学习器的加法模型。

**把上面的关系收束为本章计算式：**

$$
F_M(x)=F_0(x)+\sum_{m=1}^{M}\eta\,h_m(x)
$$

**符号解释：** $h_m$ 是第 $m$ 个弱学习器，$\eta$ 是学习率。

**代码对应：** 联合调节 `learning_rate` 与 `n_estimators`，用验证集观察过拟合。

**使用边界：** 较小学习率通常需要更多树；加法结构仍可能学习到数据偏差。


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

data = load_diabetes(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=91
)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


In [ ]:
try:
    pass
    # 请在下方填写代码：把 test_size 改成 0.20，再次切分 data
    # 提示：X、y 已在上面示例中定义，可直接复用
    # X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y, test_size=____, random_state=91)
    # 用 len() 分别打印训练集与测试集的样本数

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


**练一练**：上面示例里用 `test_size=0.25` 把数据切成了训练集和测试集。请试着把 `test_size` 改成 `0.20`，重新切分后观察每一项样本数量有什么变化。想一想：训练集变大、测试集变小时，同一个模型的"泛化表现"会更可信还是更不稳？


## 例 2｜模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import pandas as pd

rows = []
for rate, iters in [(0.03, 300), (0.06, 180), (0.1, 100)]:
    m = HistGradientBoostingRegressor(
        learning_rate=rate,
        max_iter=iters,
        max_leaf_nodes=10,
        l2_regularization=1,
        random_state=91,
    ).fit(X_train, y_train)
    rows.append(
        [
            rate,
            iters,
            m.n_iter_,
            mean_absolute_error(y_test, m.predict(X_test)),
        ]
    )
result = pd.DataFrame(
    rows, columns=["learning_rate", "max_iter", "actual_iter", "MAE"]
)
display(result.round(3))


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 同时增大学习率和模型复杂度
- 在测试集上进行大量参数尝试
- 认为 boosting 总会优于简单基线
- 没有报告训练时间和模型复杂度


## 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 102.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 102.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 102.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

理解梯度提升的加法模型，比较学习率、迭代轮数和叶节点复杂度。


### 你已经掌握

- 解释逐轮拟合残差
- 训练 HistGradientBoostingRegressor
- 理解 learning_rate 与 max_iter
- 使用早停控制过拟合


### 需要注意

- 同时增大学习率和模型复杂度
- 在测试集上进行大量参数尝试
- 认为 boosting 总会优于简单基线
- 没有报告训练时间和模型复杂度


## 参考答案


### 本章练习


In [ ]:
# 参考实现：把训练/测试比例改为 80% / 20%
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, y, test_size=0.20, random_state=91
)
train_n = len(X_train2)
test_n = len(X_test2)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
practice = []
for leaves in [5, 10, 20]:
    m = HistGradientBoostingRegressor(
        max_leaf_nodes=leaves,
        max_iter=180,
        learning_rate=0.06,
        random_state=91,
    ).fit(X_train, y_train)
    practice.append([leaves, mean_absolute_error(y_test, m.predict(X_test))])
practice_result = pd.DataFrame(practice, columns=["leaves", "MAE"])
display(practice_result)
